<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_04_Real_FHIR_R4_Preflight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
from decimal import Decimal, InvalidOperation
from importlib.metadata import version as pkg_version, PackageNotFoundError
import csv, hashlib, hmac, io, json, os, platform, re, secrets, subprocess, sys, uuid, urllib.request, zipfile
import pandas as pd
import jsonschema

UTC_NOW = datetime.now(timezone.utc).isoformat()
IS_COLAB = Path('/content').exists()
RUN = (Path('/content') if IS_COLAB else Path.cwd()) / 'ICHI2027_phase04'
PRIVATE = RUN / 'private_runtime_only'
PRIVATE.mkdir(parents=True,exist_ok=True)
DRIVE_ROOT = Path('/content/drive/MyDrive/NeuroFHIR_ICHI2027')
DRIVE_OK = False
if IS_COLAB:
    from google.colab import drive
    try:
        drive.mount('/content/drive',force_remount=False)
        DRIVE_OK = (Path('/content/drive/MyDrive').is_dir())
    except Exception as exc:
        print('Google Drive unavailable:',type(exc).__name__)
DEST = (DRIVE_ROOT/'phase04_outputs') if DRIVE_OK else (RUN/'shareable_outputs')
DEST.mkdir(parents=True,exist_ok=True)
ENV = {'python':sys.version.split()[0], 'pandas':pd.__version__,
       'jsonschema':pkg_version('jsonschema'), 'platform':platform.platform()}
print(json.dumps({'phase':'04_real_fhir_serialization_preflight','environment':ENV,
                  'drive_output_enabled':DRIVE_OK},indent=2))

Mounted at /content/drive
{
  "phase": "04_real_fhir_serialization_preflight",
  "environment": {
    "python": "3.13.15",
    "pandas": "2.2.3",
    "jsonschema": "4.26.0",
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39"
  },
  "drive_output_enabled": true
}


In [2]:
phase03_locations=[DRIVE_ROOT/'phase03_outputs'/'ICHI2027_Phase03_SHAREABLE.json',
                   Path('/content/ICHI2027_Phase03_SHAREABLE.json'),
                   RUN/'ICHI2027_Phase03_SHAREABLE.json',
                   Path('ICHI2027_Phase03_SHAREABLE.json')]
previous = next((p for p in phase03_locations if p.is_file()),None)
if previous is None and IS_COLAB:
    from google.colab import files as colab_files
    incoming = colab_files.upload()
    expected='ICHI2027_Phase03_SHAREABLE.json'
    if expected not in incoming: raise FileNotFoundError('STOP: previous Phase03 audit JSON is required.')
    previous=PRIVATE/expected
    previous.write_bytes(incoming[expected]); del incoming
if previous is None: raise FileNotFoundError('STOP: previous Phase03 audit JSON is required.')
PHASE03=json.loads(previous.read_text(encoding='utf-8'))
assert PHASE03.get('phase')=='03_verified_source_evidence','STOP: wrong previous experiment phase.'
assert PHASE03.get('scientific_results_status')=='NO_ETL_COMPARISON_RUN','STOP: unexpected Phase03 status.'
assert PHASE03['source_cross_file_overlap']['landmark_not_in_longitudinal']==0
assert PHASE03['wide_source_join']['participants_with_any_shared_column_disagreement']==0
print('Phase03 prerequisite verified; original input tables and participant identifiers are required.')

Phase03 prerequisite verified; original input tables and participant identifiers are required.


In [3]:
FILENAMES={
    'longitudinal':'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv',
    'repeat_pairs':'TRACE_ADNI1_repeat_pairs_REAL.csv',
    'landmark12':'TRACE_ADNI1_landmark12_REAL.csv',
}
SEARCH_DIRS=[Path('/content'),PRIVATE,DRIVE_ROOT/'private_inputs',DRIVE_ROOT/'inputs',
             Path('/content/ICHI2027_phase03/private_runtime_only'),
             Path('/content/ichi2027_neuroimaging_phase02/private_input'),
             Path('/content/ichi2027_neuroimaging_phase01/authorized_input')]
def sha256_file(path):
    d=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1048576),b''): d.update(block)
    return d.hexdigest()
SOURCE_PATHS={}
for role,name in FILENAMES.items():
    candidates=[d/name for d in SEARCH_DIRS if (d/name).is_file()]
    if len({sha256_file(p) for p in candidates})>1:
        raise RuntimeError('STOP: conflicting copies of '+name+' detected. Resolve before running.')
    if candidates: SOURCE_PATHS[role]=candidates[0]
if IS_COLAB:
    from google.colab import files as colab_files
    for role,name in FILENAMES.items():
        if role in SOURCE_PATHS: continue
        print('Missing authorized source:',name)
        incoming=colab_files.upload()
        if name not in incoming: raise FileNotFoundError('STOP: missing required source '+name)
        p=PRIVATE/name
        p.write_bytes(incoming[name]);SOURCE_PATHS[role]=p;del incoming
if set(SOURCE_PATHS)!=set(FILENAMES):
    raise FileNotFoundError('STOP: all three original authorized source files required.')
DATA={}
for role,path in SOURCE_PATHS.items():
    df=pd.read_csv(path,dtype=str,keep_default_na=False,low_memory=False,encoding='utf-8-sig')
    prior=PHASE03['source_profiles'][role]
    assert len(df)==prior['rows'] and len(df.columns)==prior['columns'], 'STOP: source profile differs from Phase03: '+role
    assert len(df.columns)==len(set(df.columns)),'STOP: duplicate CSV headers: '+role
    DATA[role]=df
assert DATA['longitudinal']['RID'].astype(str).str.strip().is_unique,'STOP: ambiguous longitudinal participant key.'
assert DATA['landmark12']['RID'].astype(str).str.strip().is_unique,'STOP: ambiguous landmark participant key.'
assert (set(DATA['landmark12']['RID'].str.strip()) <= set(DATA['longitudinal']['RID'].str.strip())),'STOP: failed subset relationship.'
PRIVATE_FINGERPRINTS={role:sha256_file(path) for role,path in SOURCE_PATHS.items()}
print(json.dumps({role:{'rows':len(frame),'columns':len(frame.columns)} for role,frame in DATA.items()},indent=2))

{
  "longitudinal": {
    "rows": 258,
    "columns": 51
  },
  "repeat_pairs": {
    "rows": 90,
    "columns": 51
  },
  "landmark12": {
    "rows": 199,
    "columns": 70
  }
}


In [4]:
FEATURES={
 'LHIPPOC':('hippocampus','left'), 'RHIPPOC':('hippocampus','right'),
 'LENTORHIN':('entorhinal cortex','left'), 'RENTORHIN':('entorhinal cortex','right'),
 'LMIDTEMP':('middle temporal region','left'), 'RMIDTEMP':('middle temporal region','right'),
 'VENTRICLES':('ventricles','unspecified'), 'ICV':('intracranial volume','not applicable'),
}
REQUIRED={'RID','PTID'} | {
 f'{tp}_{col}' for tp in ('BL','M12') for col in
 ('EXAMDATE','IMAGEUID','LONIUID','VERSION','OVERALLQC','TEMPQC','VENTQC',*FEATURES)
}
missing=REQUIRED-set(DATA['longitudinal'].columns)
if missing: raise ValueError('STOP: missing real source columns: '+repr(sorted(missing)))
MISSING_TOKENS={'','na','n/a','nan','none','null','unknown','unavailable','not available'}
def has_value(v): return str(v).strip().lower() not in MISSING_TOKENS
def as_decimal(raw):
    value=Decimal(str(raw).strip())
    if not value.is_finite(): raise ValueError('STOP: non-finite MRI source measurement encountered.')
    return value

def as_fhir_date(raw):
    original=str(raw).strip()
    parsed=pd.to_datetime(original,errors='coerce')
    if pd.isna(parsed): raise ValueError('STOP: unparseable real MRI acquisition date.')
    if re.fullmatch(r'\d{4}-\d{2}-\d{2}',original):return original,False
    if re.fullmatch(r'\d{4}/\d{2}/\d{2}',original):return parsed.date().isoformat(),False
    if re.search(r'\d{1,2}:\d{2}',original):
        raise ValueError('STOP: source contains acquisition time; no timezone policy has been verified.')
    return parsed.date().isoformat(),original!=parsed.date().isoformat()

assert PHASE03['evidence_contract']['participant_key'].startswith('RID')
assert PHASE03['evidence_contract']['unit'] is None
assert PHASE03['evidence_contract']['dicom_study_uid'] is None
assert PHASE03['evidence_contract']['trace_state'] is None
print('Confirmed unresolved: source units, DICOM study identity, original TRACE state. No default assigned.')

Confirmed unresolved: source units, DICOM study identity, original TRACE state. No default assigned.


In [5]:
FHIR_R4_VERSION='4.0.1'
FHIR_SCHEMA_URL='https://hl7.org/fhir/R4/fhir.schema.json.zip'
SCHEMA_FILE=RUN/'fhir_r4_schema_snapshot.zip'
SCHEMA=None
SCHEMA_REPORT={'specification':'FHIR R4 4.0.1','official_schema_url':FHIR_SCHEMA_URL,
               'status':'NOT_DOWNLOADED','schema_sha256':None,'error_type':None}
try:
    request=urllib.request.Request(FHIR_SCHEMA_URL,headers={'User-Agent':'ICHI2027-colab-research/1.0'})
    with urllib.request.urlopen(request,timeout=40) as connection: schema_bytes=connection.read()
    archive=zipfile.ZipFile(io.BytesIO(schema_bytes))
    matches=[n for n in archive.namelist() if n.endswith('fhir.schema.json')]
    if len(matches)!=1: raise ValueError('Official FHIR schema archive does not contain a unique fhir.schema.json.')
    SCHEMA=json.loads(archive.read(matches[0]))
    if not isinstance(SCHEMA.get('definitions'),dict) or 'Bundle' not in SCHEMA['definitions']:
        raise ValueError('Official schema missing necessary resource definitions.')
    SCHEMA_REPORT.update(status='DOWNLOADED_UNVALIDATED',schema_sha256=hashlib.sha256(schema_bytes).hexdigest(),
                         schema_member=matches[0],definitions_count=len(SCHEMA['definitions']))
except Exception as exc:
    SCHEMA=None
    SCHEMA_REPORT.update(status='UNAVAILABLE_OFFICIAL_R4_VALIDATION_NOT_POSSIBLE',error_type=type(exc).__name__)
print(json.dumps(SCHEMA_REPORT,indent=2))

{
  "specification": "FHIR R4 4.0.1",
  "official_schema_url": "https://hl7.org/fhir/R4/fhir.schema.json.zip",
  "status": "DOWNLOADED_UNVALIDATED",
  "schema_sha256": "75e5560da3cf503895a44c8ca7af17a83b4cca6c2cb5ba1883d2aec0d1cb5ac6",
  "error_type": null,
  "schema_member": "fhir.schema.json",
  "definitions_count": 680
}


In [6]:
PRIVATE_PERSIST=(DRIVE_ROOT/'_private_reproducibility') if DRIVE_OK else PRIVATE
PRIVATE_PERSIST.mkdir(parents=True,exist_ok=True)
KEY_PATH=PRIVATE_PERSIST/'ich i2027_pseudonymization_key.hex'.replace(' ','')
KEY_REUSED=KEY_PATH.is_file()
if KEY_REUSED:
    SECRET=bytes.fromhex(KEY_PATH.read_text(encoding='ascii').strip())
    if len(SECRET)!=32: raise RuntimeError('STOP: unexpected private pseudonymization key length.')
else:
    SECRET=secrets.token_bytes(32)
    KEY_PATH.write_text(SECRET.hex(),encoding='ascii')
    try: KEY_PATH.chmod(0o600)
    except OSError:pass

def opaque(*parts):
    return hmac.new(SECRET,'|'.join(map(str,parts)).encode('utf-8'),hashlib.sha256).hexdigest()
def bundle_uuid(*parts):return 'urn:uuid:'+str(uuid.UUID(hex=opaque(*parts)[:32]))
def local_resource_id(prefix,*parts):return prefix+'-'+opaque(*parts)[:30]
SOURCE_IDENTIFIER_SYSTEMS={
 'IMAGEUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-imageuid',
 'LONIUID':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/source-identifier/adni-loniuid',
}
BUNDLES=[]
PRIVATE_LEDGER=[]
BUILD_ERRORS=Counter()
DATE_NORMALIZATIONS=Counter()
for rec in DATA['longitudinal'].itertuples(index=False,name=None):
    row=dict(zip(DATA['longitudinal'].columns,rec))
    rid=str(row['RID']).strip()
    if not has_value(rid):raise ValueError('STOP: blank source participant RID.')
    p_url=bundle_uuid('Patient',rid)
    patient={'resourceType':'Patient','id':local_resource_id('p','Patient',rid),
             'identifier':[{'system':'https://github.com/SANGHATI23/neuroimaging-fhir-omop-fidelity/local-pseudonym',
                            'value':opaque('research-only-pseudonym',rid)}]}
    entries=[{'fullUrl':p_url,'resource':patient}]
    for tp in ('BL','M12'):
        dt,normalized=as_fhir_date(row[f'{tp}_EXAMDATE'])
        if normalized:DATE_NORMALIZATIONS[tp]+=1
        img_url=bundle_uuid('ImagingStudy',rid,tp)
        identifiers=[]
        for field in ('IMAGEUID','LONIUID'):
            raw_id=str(row[f'{tp}_{field}']).strip()
            if has_value(raw_id):
                identifiers.append({'system':SOURCE_IDENTIFIER_SYSTEMS[field],'value':raw_id})
        if not identifiers: raise ValueError('STOP: source image references missing for a required timepoint.')
        img={'resourceType':'ImagingStudy','id':local_resource_id('im','ImagingStudy',rid,tp),
             'status':'unknown','subject':{'reference':p_url},
             'started':dt,'identifier':identifiers}
        entries.append({'fullUrl':img_url,'resource':img})
        for feature,(region,laterality) in FEATURES.items():
            source_col=f'{tp}_{feature}'
            source_value=str(row[source_col]).strip()
            if not has_value(source_value):
                BUILD_ERRORS[f'missing_{source_col}']+=1;continue
            try:value=as_decimal(source_value)
            except (InvalidOperation,ValueError):
                BUILD_ERRORS[f'invalid_{source_col}']+=1;continue
            if abs(value)>Decimal('1e100'):
                BUILD_ERRORS[f'unsupported_fhir_numeric_magnitude_{source_col}']+=1;continue
            observation_url=bundle_uuid('Observation',rid,tp,feature)
            anatomy=f'{laterality} {region}'.strip() if laterality not in ('unspecified','not applicable') else region
            observation={
              'resourceType':'Observation','id':local_resource_id('o','Observation',rid,tp,feature),
              'status':'unknown','category':[{'coding':[{
                  'system':'http://terminology.hl7.org/CodeSystem/observation-category','code':'imaging'}]}],
              'code':{'text':f'{anatomy} volume; source field {source_col}'},
              'subject':{'reference':p_url},'effectiveDateTime':dt,
              'derivedFrom':[{'reference':img_url}],
              'bodySite':{'text':anatomy},
              'valueQuantity':{'value':float(value)},
            }
            entries.append({'fullUrl':observation_url,'resource':observation})
            PRIVATE_LEDGER.append({
               'observation_full_url':observation_url,'patient_full_url':p_url,'imaging_full_url':img_url,
               'value_decimal':value,'source_column':source_col,'source_region':region,
               'source_laterality':laterality,'source_date_fhir':dt,
               'source_version_uninterpreted':str(row[f'{tp}_VERSION']).strip(),
               'source_qc_uninterpreted':{k:str(row[f'{tp}_{k}']).strip() for k in ('OVERALLQC','TEMPQC','VENTQC')},
               'unit_verified':False,'original_trace_state_available':False,
            })
    BUNDLES.append({'resourceType':'Bundle','type':'collection','entry':entries})
print(json.dumps({'source_rows_processed':len(BUNDLES),'real_observations_encoded':len(PRIVATE_LEDGER),
                  'source_value_exceptions':dict(BUILD_ERRORS),'normalized_date_strings':dict(DATE_NORMALIZATIONS)},indent=2))

{
  "source_rows_processed": 258,
  "real_observations_encoded": 4128,
  "source_value_exceptions": {},
  "normalized_date_strings": {}
}


In [7]:
LOCAL_CHECKS=Counter()
SOURCE_COUNTS=Counter()
if len(BUNDLES)!=len(DATA['longitudinal']):raise AssertionError('STOP: participant bundle cardinality changed.')
ledger_index={r['observation_full_url']:r for r in PRIVATE_LEDGER}
seen_fullurls=set()
for bundle in BUNDLES:
    inside={e['fullUrl']:e['resource'] for e in bundle['entry']}
    assert len(inside)==len(bundle['entry']),'STOP: duplicate FHIR fullUrl in a bundle.'
    for link,resource in inside.items():
        if link in seen_fullurls: raise AssertionError('STOP: duplicate FHIR fullUrl across participants.')
        seen_fullurls.add(link)
        typ=resource['resourceType'];SOURCE_COUNTS[typ]+=1
        if typ=='Patient':continue
        if resource['subject']['reference'] not in inside:LOCAL_CHECKS['unresolved_patient_reference']+=1
        if typ=='ImagingStudy':
            if not resource.get('identifier'):LOCAL_CHECKS['imaging_missing_source_identifiers']+=1
            if 'uid' in resource:LOCAL_CHECKS['unverified_dicom_study_uid_claim']+=1
        if typ=='Observation':
            lid=resource['derivedFrom'][0]['reference']
            if lid not in inside or inside[lid]['resourceType']!='ImagingStudy':
                LOCAL_CHECKS['unresolved_imaging_reference']+=1
            if link not in ledger_index:LOCAL_CHECKS['observation_without_source_evidence']+=1
            else:
                original=ledger_index[link]
                actual=Decimal(str(resource['valueQuantity']['value']))
                if abs(actual-original['value_decimal'])>Decimal('0.000000001'):
                    LOCAL_CHECKS['measurement_value_mismatch']+=1
                if resource['effectiveDateTime']!=original['source_date_fhir']:
                    LOCAL_CHECKS['acquisition_date_mismatch']+=1
                if original['source_region'] not in resource['bodySite']['text']:
                    LOCAL_CHECKS['anatomical_region_mismatch']+=1
                if original['source_laterality'] not in ('unspecified','not applicable') and (
                   original['source_laterality'] not in resource['bodySite']['text']):
                    LOCAL_CHECKS['laterality_mismatch']+=1
            if 'unit' in resource['valueQuantity'] or 'code' in resource['valueQuantity']:
                LOCAL_CHECKS['unverified_unit_inferred']+=1
if len(ledger_index)!=SOURCE_COUNTS['Observation']:
    LOCAL_CHECKS['event_cardinality_mismatch']+=abs(len(ledger_index)-SOURCE_COUNTS['Observation'])
if LOCAL_CHECKS:raise AssertionError('STOP: local source-to-FHIR checks failed: '+repr(dict(LOCAL_CHECKS)))
print(json.dumps({'real_fhir_resources':dict(SOURCE_COUNTS),'reference_and_value_checks':'PASS',
                  'quantity_units_assigned':False,'dicom_study_uids_assigned':False},indent=2))

{
  "real_fhir_resources": {
    "Patient": 258,
    "ImagingStudy": 516,
    "Observation": 4128
  },
  "reference_and_value_checks": "PASS",
  "quantity_units_assigned": false,
  "dicom_study_uids_assigned": false
}


In [8]:
R4_VALIDATION={'status':'NOT_RUN_OFFICIAL_SCHEMA_UNAVAILABLE','schema_sha256':SCHEMA_REPORT['schema_sha256'],
               'resource_type_validations':{},'validation_failures_by_type':{},'all_generated_resources_validated':False}
if SCHEMA is not None:
    definitions=SCHEMA['definitions']
    validators={}
    for kind in ('Bundle','Patient','ImagingStudy','Observation'):
        if kind not in definitions: raise ValueError('STOP: official FHIR R4 schema lacks '+kind)
        wrapper={'$schema':SCHEMA.get('$schema','http://json-schema.org/draft-06/schema#'),
                 'definitions':definitions,'$ref':'#/definitions/'+kind}
        validators[kind]=jsonschema.validators.validator_for(wrapper)(wrapper)
    validated=Counter();failed=Counter();first_error={}
    for bundle in BUNDLES:
        for typ,payload in [('Bundle',bundle)]+[(x['resource']['resourceType'],x['resource']) for x in bundle['entry']]:
            errors=list(validators[typ].iter_errors(payload))
            validated[typ]+=1
            if errors:
                failed[typ]+=1
                if typ not in first_error:
                    # Schema errors alone; no patient identifier, values, or raw FHIR resources included.
                    first_error[typ]={'validator':str(errors[0].validator),
                                      'schema_path':[str(x) for x in errors[0].schema_path],
                                      'instance_path':[str(x) for x in errors[0].path]}
    R4_VALIDATION={'status':'PASS' if not failed else 'FAILED_OFFICIAL_SCHEMA',
                   'schema_sha256':SCHEMA_REPORT['schema_sha256'],
                   'resource_type_validations':dict(validated),
                   'validation_failures_by_type':dict(failed),
                   'first_failure_schema_paths_only':first_error,
                   'all_generated_resources_validated':not bool(failed)}
print(json.dumps(R4_VALIDATION,indent=2))

{
  "status": "PASS",
  "schema_sha256": "75e5560da3cf503895a44c8ca7af17a83b4cca6c2cb5ba1883d2aec0d1cb5ac6",
  "resource_type_validations": {
    "Bundle": 258,
    "Patient": 258,
    "ImagingStudy": 516,
    "Observation": 4128
  },
  "validation_failures_by_type": {},
  "first_failure_schema_paths_only": {},
  "all_generated_resources_validated": true
}


In [9]:
UPSTREAM={
 'imagewg':{'url':'https://github.com/OHDSI/ImageWG.git',
            'commit':PHASE03['upstream_revisions']['imagewg']['expected_commit']},
 'ohdsi_tfl':{'url':'https://github.com/SANGHATI23/ohdsi-fhir-omop-showcase-demo.git',
             'commit':PHASE03['upstream_revisions']['ohdsi_tfl']['expected_commit']},
}
PINNED={}
for name,repo in UPSTREAM.items():
    folder=RUN/'upstream'/name
    try:
        if not (folder/'.git').is_dir():
            folder.mkdir(parents=True,exist_ok=True)
            subprocess.run(['git','init','-q',str(folder)],check=True,capture_output=True,timeout=30)
            subprocess.run(['git','-C',str(folder),'remote','add','origin',repo['url']],check=True,capture_output=True,timeout=30)
        subprocess.run(['git','-C',str(folder),'fetch','-q','--depth','1','origin',repo['commit']],check=True,capture_output=True,timeout=120)
        subprocess.run(['git','-C',str(folder),'checkout','-q','--detach','FETCH_HEAD'],check=True,capture_output=True,timeout=30)
        actual=subprocess.run(['git','-C',str(folder),'rev-parse','HEAD'],check=True,capture_output=True,text=True,timeout=10).stdout.strip()
        PINNED[name]={'status':'PINNED_AND_VERIFIED' if actual==repo['commit'] else 'COMMIT_MISMATCH',
                      'commit':actual}
    except (subprocess.CalledProcessError,subprocess.TimeoutExpired) as exc:
        PINNED[name]={'status':'UNAVAILABLE_THIS_RUN','error_type':type(exc).__name__}
print(json.dumps(PINNED,indent=2))

{
  "imagewg": {
    "status": "PINNED_AND_VERIFIED",
    "commit": "8f0a19bb94e941bb335a097bc9851fab672621c7"
  },
  "ohdsi_tfl": {
    "status": "PINNED_AND_VERIFIED",
    "commit": "a4b7265adabf096e84b957de068e24a5ab853425"
  }
}


In [10]:
MI_SPEC={'status':'NO_VERIFIED_PINNED_FIELD_SPEC','field_definitions':{},'required_fields_by_table':{}}
file=RUN/'upstream'/'imagewg'/'inst/csv/new_csv/OMOP_CDM_Medical_Imaging_Ex_Field_Level.csv'
if PINNED.get('imagewg',{}).get('status')=='PINNED_AND_VERIFIED' and file.is_file():
    with file.open(encoding='utf-8-sig',newline='') as handle:
        reader=csv.DictReader(handle)
        raw_rows=list(reader)
    def norm_key(s):return re.sub(r'[^a-z]','',s.lower())
    fields={norm_key(k):k for k in (reader.fieldnames or [])}
    table_col=fields.get('cdmtablename');name_col=fields.get('cdmfieldname')
    required_col=fields.get('isrequired');dtype_col=fields.get('cdmdatatype')
    if not (table_col and name_col):raise ValueError('STOP: pinned MI-CDM CSV header changed unexpectedly.')
    field_defs={};required={}
    for row in raw_rows:
        table=str(row.get(table_col,'')).strip().upper()
        field=str(row.get(name_col,'')).strip()
        if table not in ('IMAGE_OCCURRENCE','IMAGE_FEATURE') or not field:continue
        is_required=str(row.get(required_col,'')).strip().lower() if required_col else ''
        item={'field':field,'datatype_as_specified':str(row.get(dtype_col,'')).strip() if dtype_col else '',
              'is_required_as_specified':is_required}
        field_defs.setdefault(table,[]).append(item)
        if is_required in ('true','1','yes','y','required','x'):
            required.setdefault(table,[]).append(field)
    MI_SPEC={'status':'PINNED_FIELD_SPEC_ONLY_NO_EXECUTABLE_DDL',
             'field_spec_sha256':sha256_file(file),
             'field_definitions':field_defs,'required_fields_by_table':required,
             'schema_executable':False,'vocabulary_pinned':False}
print(json.dumps({k:v for k,v in MI_SPEC.items() if k!='field_definitions'},indent=2))

{
  "status": "PINNED_FIELD_SPEC_ONLY_NO_EXECUTABLE_DDL",
  "field_spec_sha256": "af2e1d26aae56d05f3cc5d90f5ac3d523b64a15dcb4e8ee038a5c2a0aa129cf4",
  "required_fields_by_table": {
    "IMAGE_OCCURRENCE": [
      "image_occurrence_id",
      "person_id",
      "procedure_occurrence_id",
      "image_occurrence_date",
      "image_study_UID",
      "image_series_UID",
      "modality_concept_id"
    ],
    "IMAGE_FEATURE": [
      "image_feature_id",
      "person_id",
      "image_occurrence_id",
      "image_feature_concept_id",
      "image_feature_type_concept_id"
    ]
  },
  "schema_executable": false,
  "vocabulary_pinned": false
}


In [11]:
TFL_INVENTORY={'status':'NOT_AVAILABLE_THIS_RUN','original_tfl_v15_executed':False}
folder=RUN/'upstream'/'ohdsi_tfl'
if PINNED.get('ohdsi_tfl',{}).get('status')=='PINNED_AND_VERIFIED':
    candidates=[]
    token=re.compile(r'\b(?:PRESERVED|NORMALIZED|COLLAPSED|AMBIGUOUS|TRACEABILITY_LOSS|UNMAPPED|TFL)\b',re.I)
    permitted={'.py','.ipynb','.sql','.json','.md','.yml','.yaml'}
    for p in folder.rglob('*'):
        if not p.is_file() or '.git' in p.parts or p.suffix.lower() not in permitted:continue
        if p.stat().st_size>2500000:continue
        body=p.read_text(encoding='utf-8',errors='replace')
        hits=len(token.findall(body))
        if hits:
            candidates.append({'path':p.relative_to(folder).as_posix(),
                               'sha256':sha256_file(p),'term_occurrences':hits,
                               'file_type':p.suffix.lower()})
    candidates=sorted(candidates,key=lambda x:(-x['term_occurrences'],x['path']))
    TFL_INVENTORY={'status':'STATIC_CODE_AND_DOCUMENTATION_CANDIDATES_ONLY',
                   'candidates':candidates,'original_tfl_v15_executed':False}
print(json.dumps({'status':TFL_INVENTORY['status'],
                  'candidate_files':len(TFL_INVENTORY.get('candidates',[])),
                  'original_tfl_v15_executed':False},indent=2))

{
  "status": "STATIC_CODE_AND_DOCUMENTATION_CANDIDATES_ONLY",
  "candidate_files": 14,
  "original_tfl_v15_executed": false
}


In [12]:
if R4_VALIDATION['status']=='FAILED_OFFICIAL_SCHEMA':
    FHIR_STATUS='FAILED_OFFICIAL_SCHEMA_REQUIRES_DEBUGGING'
elif R4_VALIDATION['status']=='PASS':
    FHIR_STATUS='STRUCTURAL_R4_PASS_REAL_SOURCE_UNITS_UNVERIFIED'
else:
    FHIR_STATUS='LOCAL_STRUCTURAL_PASS_OFFICIAL_R4_SCHEMA_NOT_RUN'
REPORT={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'04_real_fhir_serialization_preflight',
 'run_utc':UTC_NOW,
 'scientific_results_status':'REAL_SOURCE_SERIALIZATION_ONLY_NO_ETL_COMPARISON',
 'privacy':'Aggregate schema, validation and contract metadata only; no raw IDs, patient rows, FHIR bundles or source CSV bytes.',
 'environment':ENV,
 'source_profile_check':{role:{'rows':len(df),'columns':len(df.columns)} for role,df in DATA.items()},
 'phase03_prerequisite_loaded':True,
 'source_patient_key':'RID local-only; reusable private HMAC key never included in shared outputs',
 'pseudonymization_key_reused_from_private_store':KEY_REUSED,
 'private_reproducibility_key_store_enabled':DRIVE_OK,
 'real_fhir_in_memory':{'status':FHIR_STATUS,'bundle_count':len(BUNDLES),
                       'resource_type_counts':dict(SOURCE_COUNTS),
                       'source_to_fhir_invariant_failures':dict(LOCAL_CHECKS),
                       'unencoded_or_invalid_measurements_by_source_column':dict(BUILD_ERRORS),
                       'source_date_normalizations':dict(DATE_NORMALIZATIONS),
                       'FHIR_R4_version_requested':FHIR_R4_VERSION,
                       'has_source_verified_quantity_units':False,
                       'has_verified_dicom_study_uid':False,
                       'source_qc_and_version_in_private_ledger_only':True,
                       'clinical_status_asserted':False,
                       'repeat_pairs_transformed':False},
 'official_fhir_schema':SCHEMA_REPORT,
 'official_r4_validation':R4_VALIDATION,
 'mi_cdm_pinned_field_spec':MI_SPEC,
 'original_tfl_v15_static_inventory':TFL_INVENTORY,
 'pinned_repositories':PINNED,
 'experiment_gates':{
   'real_source_fhir_serialization_completed':bool(len(BUNDLES)),
   'referential_and_value_checks_passed':not bool(LOCAL_CHECKS),
   'official_FHIR_R4_schema_validation_passed':R4_VALIDATION['status']=='PASS',
   'unit_confirmed_by_authoritative_source':False,
   'original_trace_state_and_calibration_recovered':False,
   'dicom_study_uid_verified':False,
   'original_tfl_v15_executed':False,
   'omop_vocabulary_snapshot_pinned':False,
   'mi_cdm_target_loaded':False,
   'real_four_arm_comparison_performed':False,
 },
 'comparative_arm_scores':None,
 'cohort_equivalence_statistics':None,
 'next_stage_gate':'AUTHORITATIVE_UNIT_AND_PROVENANCE_PLUS_VALIDATED_R4_BEFORE_REAL_MICDM_LOAD',
}
PRIVATE_MANIFEST=PRIVATE_PERSIST/'ICHI2027_Phase04_PRIVATE_MANIFEST.json'
PRIVATE_MANIFEST.write_text(json.dumps({'phase':'04_real_fhir_serialization_preflight',
                                       'run_utc':UTC_NOW,'source_csv_sha256_private':PRIVATE_FINGERPRINTS,
                                       'phase03_audit_sha256':sha256_file(previous),
                                       'FHIR_R4_schema_sha256':SCHEMA_REPORT['schema_sha256'],
                                       'private_key_file_name':KEY_PATH.name,
                                       'key_material_in_manifest':False},indent=2),encoding='utf-8')
try: PRIVATE_MANIFEST.chmod(0o600)
except OSError:pass
OUTPUT=DEST/'ICHI2027_Phase04_SHAREABLE.json'
OUTPUT.write_text(json.dumps(REPORT,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
assert all(x not in OUTPUT.read_text(encoding='utf-8') for x in ('PTID','BL_IMAGEUID','M12_IMAGEUID','source_imageuid_raw'))
print(json.dumps({'saved_aggregate_output':str(OUTPUT),'official_fhir_status':FHIR_STATUS,
                  'research_comparator_scores_generated':False,
                  'csv_or_participant_level_fhir_exported_to_drive':False},indent=2))

{
  "saved_aggregate_output": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase04_outputs/ICHI2027_Phase04_SHAREABLE.json",
  "official_fhir_status": "STRUCTURAL_R4_PASS_REAL_SOURCE_UNITS_UNVERIFIED",
  "research_comparator_scores_generated": false,
  "csv_or_participant_level_fhir_exported_to_drive": false
}


In [13]:
if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(str(OUTPUT))
else:
    print('Shareable Phase04 output:',str(OUTPUT))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>